# Build a Basic Chatbot with Langgraph (GRAPH API)

In [ ]:
from typing_extensions import TypedDict
from typing import Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

In [ ]:
provider = "ollama"
model = "qwen3-0.6b:latest"

In [ ]:
class State(TypedDict):
    # Messages are list. The add_function in the annotation
    # defines how this state key should be updated.
    # (here it appends messages to the list, not overwritting!)
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

graph_builder

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model=model)

llm

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(f"{provider}:{model}")

llm

In [ ]:
# Node functionality
def chatbot(state:State):
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
graph_builder = StateGraph(State)

# adding node
graph_builder.add_node("llm_chatbot", chatbot)

# adding edges
graph_builder.add_edge(START, "llm_chatbot")
graph_builder.add_edge("llm_chatbot", END)

# compile the graph
graph = graph_builder.compile()

In [ ]:
# Visualize the graph
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
response = graph.invoke({"messages": "HI!"})

In [ ]:
response["messages"][-1].content

In [ ]:
for event in graph.stream({"messages": "Hi, How are you?"}):
    print(event, "\n", "-"*20)
    for value in event.values():
        print(value, "\n", "+"*20)
        print(value['messages'][-1].content)

## Chatbot with Tool

In [ ]:
# Custom functions
def multiply(a:int, b:int) -> int:
    """
    Multiply a and b

    Args:
        a (int): first integer
        b (int): second integer

    Returns:
        int: output integer
    """

    return a * b

def sum(a:int, b:int) -> int:
    """
    Sum a and b

    Args:
        a (int): first integer
        b (int): second integer

    Returns:
        int: output integer
    """

    return a + b

def check_calendar(day:str) -> str:
    """
    Check the day

    Args:
        day (str): the day of week string

    Returns:
        str: output string
    """
    
    temp = {
        "Saturday": "Doctor 10am, Meeting 2pm",
        "Monday": "Class 12am"
        }
    return temp.get(day, default="Nothing")

tools = [multiply, sum, check_calendar]

In [ ]:
from langchain.chat_models import init_chat_model

provider = "ollama"
model = "qwen3-0.6b:latest"

llm = init_chat_model(f"{provider}:{model}")
llm_with_tool = llm.bind_tools(tools=tools)

llm_with_tool

In [ ]:
# StateGraph
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    # Messages are list. The add_function in the annotation
    # defines how this state key should be updated.
    # (here it appends messages to the list, not overwritting!)
    messages: Annotated[list, add_messages]

# Node definition
def tool_calling_llm(state:State):
    return {"messages": [llm_with_tool.invoke(state["messages"])]}

# Graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

# Add edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant 
    # is a tool call -> tools_condition routes to tools
    # and if is a not a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools", END)

# Compile the graph
graph = builder.compile()

In [ ]:
# Visualize the graph
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
graph.invoke({"messages": "What is the 3*7?"})

In [ ]:
response = graph.invoke({"messages": "What is the 37+84?"})

response["messages"][-1].content

In [ ]:
for msg in response["messages"]:
    msg.pretty_print()

In [ ]:
response = graph.invoke({"messages": "What is the 3+2*7?"})

response["messages"][-1].content

In [ ]:
for msg in response["messages"]:
    msg.pretty_print()

## ReAct Agent

In [ ]:
# StateGraph
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict
from typing import Annotated

class State(TypedDict):
    # Messages are list. The add_function in the annotation
    # defines how this state key should be updated.
    # (here it appends messages to the list, not overwritting!)
    messages: Annotated[list, add_messages]

# Node definition
def tool_calling_llm(state:State):
    return {"messages": [llm_with_tool.invoke(state["messages"])]}

# Graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

# Add edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant 
    # is a tool call -> tools_condition routes to tools
    # and if is a not a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools", "tool_calling_llm")

# Compile the graph
graph = builder.compile()

In [ ]:
# Visualize the graph
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
response = graph.invoke({"messages": "Today is Monday, see my calendar and say What is the 3+2*7?"})

response["messages"][-1].content

In [ ]:
for msg in response["messages"]:
    msg.pretty_print()

## Adding Memory in Agent Graph

In [ ]:
response = graph.invoke({"messages": "Hello my name is Wreza."})

for msg in response['messages']:
    msg.pretty_print()

In [ ]:
response = graph.invoke({"messages": "What is my name?"})

for msg in response['messages']:
    msg.pretty_print()

In [ ]:
# StateGraph
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages

memory = MemorySaver()

class State(TypedDict):
    # Messages are list. The add_function in the annotation
    # defines how this state key should be updated.
    # (here it appends messages to the list, not overwritting!)
    messages: Annotated[list, add_messages]

# Node definition
def tool_calling_llm(state:State):
    return {"messages": [llm_with_tool.invoke(state["messages"])]}

# Graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

# Add edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message (result) from assistant 
    # is a tool call -> tools_condition routes to tools
    # and if is a not a tool call -> tools_condition routes to END
    tools_condition
)
builder.add_edge("tools", "tool_calling_llm")

# Compile the graph
graph = builder.compile(checkpointer=memory)

In [ ]:
# Visualize the graph
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-1"}}

response = graph.invoke({
    "messages": [HumanMessage(content="Hi, my name is Wreza!")]},
    config=config)

for msg in response["messages"]:
    msg.pretty_print()

In [ ]:
response = graph.invoke({
    "messages": [HumanMessage(content="What is my name?")]},
    config=config)

for msg in response['messages']:
    msg.pretty_print()

### Streaming

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

In [ ]:
def superbot(state:State):
    return {"messages": [llm.invoke(state["messages"])]}

In [ ]:
graph = StateGraph(State)

graph.add_node("SuperBot", superbot)

graph.add_edge(START, "SuperBot")
graph.add_edge("SuperBot", END)

graph_builder = graph.compile(checkpointer=memory)

In [ ]:
# Visualize the graph
from IPython.display import Image, display

try:
    display(Image(graph_builder.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
config = {"configurable": {"thread_id": "test-2"}}

response = graph_builder.invoke({
    "messages": [HumanMessage(content="Hi, my name is Wreza And I like videogame")]},
    config=config)

response

In [ ]:
config = {"configurable": {"thread_id": "test03"}}
msg = [HumanMessage(content="Hi, my name is Wreza And I like videogame")]

for chunk in graph_builder.stream({"messages": msg}, config=config, stream_mode="values"):
    print(chunk)

In [ ]:
msg = [HumanMessage(content="I also like football")]

for chunk in graph_builder.stream({"messages": msg}, config=config, stream_mode="updates"):
    print(chunk)

In [ ]:
msg = [HumanMessage(content="and I like movies")]

for chunk in graph_builder.stream({"messages": msg}, config=config, stream_mode="values"):
    print(chunk)

In [ ]:
state = graph_builder.get_state(config)

print("Number of messages:", len(state.values["messages"]))

for i, message in enumerate(state.values["messages"]):
    print(i, type(message).__name__, str(message.content))